In [1]:
# Import all necessary libraries for data processing, modeling, and evaluation

import pandas as pd
import numpy as np
import re
import nltk
import pickle

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report


In [2]:
# Download required NLTK datasets for text preprocessing

nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [4]:
# Load the spam email dataset into a DataFrame

# Dataset Link
!wget -O spam.csv https://raw.githubusercontent.com/mohitgupta-1O1/Kaggle-SMS-Spam-Collection-Dataset-/master/spam.csv

df = pd.read_csv("spam.csv", encoding="latin-1")
df = df[['v1', 'v2']]
df.columns = ['label', 'text']


--2026-02-04 01:51:46--  https://raw.githubusercontent.com/mohitgupta-1O1/Kaggle-SMS-Spam-Collection-Dataset-/master/spam.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 503663 (492K) [application/octet-stream]
Saving to: ‘spam.csv’

spam.csv            100%[===================>] 491.86K  --.-KB/s    in 0.04s   

2026-02-04 01:51:46 (12.6 MB/s) - ‘spam.csv’ saved [503663/503663]



In [6]:
# Display basic information and sample rows from the dataset

df.head()


,label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [7]:
# Convert text labels (spam/ham) into numerical values

df['label'] = df['label'].map({'ham': 0, 'spam': 1})


In [8]:
# Initialize stopwords list and lemmatizer

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()


In [9]:
# Define a function to clean and preprocess email text

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words]
    return ' '.join(tokens)


In [10]:
# Apply text preprocessing to the entire dataset

df['clean_text'] = df['text'].apply(preprocess_text)


In [11]:
# Split the dataset into training and validation sets

X_train, X_val, y_train, y_val = train_test_split(
    df['clean_text'],
    df['label'],
    test_size=0.2,
    random_state=42
)


In [12]:
# Transform text data into numerical features using TF-IDF

vectorizer = TfidfVectorizer(max_features=3000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_val_tfidf = vectorizer.transform(X_val)


In [13]:
# Train a Multinomial Naive Bayes classifier

model = MultinomialNB()
model.fit(X_train_tfidf, y_train)


MultinomialNB()

In [14]:
# Predict labels for the validation dataset

y_pred = model.predict(X_val_tfidf)


In [15]:
# Evaluate the model using accuracy and classification report

print("Accuracy:", accuracy_score(y_val, y_pred))
print(classification_report(y_val, y_pred))


Accuracy: 0.9748878923766816
              precision    recall  f1-score   support

           0       0.97      1.00      0.99       965
           1       1.00      0.81      0.90       150

    accuracy                           0.97      1115
   macro avg       0.99      0.91      0.94      1115
weighted avg       0.98      0.97      0.97      1115



In [16]:
# Save the trained model and TF-IDF vectorizer for future use

with open("spam_classifier_model.pkl", "wb") as model_file:
    pickle.dump(model, model_file)

with open("tfidf_vectorizer.pkl", "wb") as vectorizer_file:
    pickle.dump(vectorizer, vectorizer_file)


In [17]:
# Load the saved model and vectorizer from disk

with open("spam_classifier_model.pkl", "rb") as model_file:
    loaded_model = pickle.load(model_file)

with open("tfidf_vectorizer.pkl", "rb") as vectorizer_file:
    loaded_vectorizer = pickle.load(vectorizer_file)


In [18]:
# Define a function to predict whether a new email is spam or ham

def predict_email(email_text):
    cleaned_text = preprocess_text(email_text)
    vectorized_text = loaded_vectorizer.transform([cleaned_text])
    prediction = loaded_model.predict(vectorized_text)
    return "Spam" if prediction[0] == 1 else "Ham"


In [23]:
# Test the spam classifier with a sample email

sample_emails = ["Congratulations! You have won a free prize.", "Congratulations! You have won a free iPhone, click now", "Are we meeting tomorrow at 5?"]
for email in sample_emails:
    print(f"Email: '{email}' --> Prediction: {predict_email(email)}")

Email: 'Congratulations! You have won a free prize.' --> Prediction: Spam
Email: 'Congratulations! You have won a free iPhone, click now' --> Prediction: Spam
Email: 'Are we meeting tomorrow at 5?' --> Prediction: Ham
